# View intensity statistics

Visualises the per-FOV intensity statistics produced by the FOV scheduler (nb05).

For each **bit** (color channel in each hybridisation round) two plots are generated:

1. **Z-profile** — median pixel intensity vs z position; one line per FOV (same color, α = 0.5).
2. **FOV heatmap** — spatial map of median intensity; FOV stage positions are converted to a regular (x_idx, y_idx) grid.

The bead channel (488 nm) and bead-plane frames (z = 0) are excluded automatically.

## 1 — Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/after_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.progress        import ProgressTracker
from MERci.acquisition.configs import find_frame_table_for_hal_config

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Experiment parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX = ".zarr"          # must match what was used in nb05

# ── Plotting parameters ────────────────────────────────────────────────
# Which rounds to show (None = all rounds with completed data)
SHOW_ROUNDS  = None

# Which color channels to show in nm (None = all non-bead channels)
# e.g. [560, 650, 750] to show only the MERFISH bits channels
SHOW_COLORS  = None

# Z value used for bead/fiducial frames — these are excluded from plots
BEAD_Z       = 0.0

# Wavelength of the fiducial bead channel — excluded from plots
BEAD_COLOR   = 488.0

print(f"Sample name  : {SAMPLE_NAME}")
print(f"Positions tag: {POSITIONS_TAG}")
print(f"Image suffix : {IMAGE_SUFFIX}")

In [ ]:
config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                   config.data_dir, image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)

print(f"Rounds : {meta.n_rounds}")
print(f"FOVs   : {meta.n_fovs}")

## 3 — Load stats data

Reads all completed stats CSVs and merges them with frame-table information (color, z position).
Results are stored in `stats_df` — a single DataFrame with one row per (FOV, frame).

In [ ]:
def _load_frame_table(config, meta, round_id):
    """Return the frame table DataFrame for *round_id*, or None if unavailable."""
    if config.settings_dir is None:
        return None
    for s in meta.series_for_round(round_id):
        if not s.hal_config:
            continue
        hal_path = config.settings_dir / s.hal_config
        ft_path  = find_frame_table_for_hal_config(hal_path, config.metadata_dir)
        if ft_path and ft_path.exists():
            return pd.read_csv(ft_path, index_col=0)
    return None


def load_stats_with_annotations(config, meta, tracker):
    """
    Load all completed stats CSVs and annotate with round_id, fov_id,
    stage position, z, and color.
    """
    ft_cache = {}   # round_id → frame table (or None)
    records  = []

    for round_id in meta.valid_round_ids():
        if round_id not in ft_cache:
            ft_cache[round_id] = _load_frame_table(config, meta, round_id)
        ft = ft_cache[round_id]

        # Build frame-info lookup (frame → color, z)
        if ft is not None:
            frame_info = (
                ft[["color", "z"]]
                .reset_index()
                .rename(columns={ft.index.name or "index": "frame"})
            )
        else:
            frame_info = None

        round_obj = meta.rounds.get(round_id)
        if round_obj is None:
            continue

        for fov_id, file_list in round_obj.fov_files.items():
            for fpath in file_list:
                sp = tracker.stats_path(fpath)
                if not sp.exists():
                    continue

                df = pd.read_csv(sp)
                df["round_id"]   = round_id
                df["fov_id"]     = fov_id
                df["position_x"] = meta.fovs[fov_id].position[0]
                df["position_y"] = meta.fovs[fov_id].position[1]

                if frame_info is not None:
                    df = df.merge(frame_info, on="frame", how="left")

                records.append(df)

    if not records:
        return pd.DataFrame()
    return pd.concat(records, ignore_index=True)


stats_df = load_stats_with_annotations(config, meta, tracker)

if stats_df.empty:
    print("No stats data found. Run nb05 first to generate stats CSVs.")
else:
    n_fovs_loaded = stats_df.groupby(["round_id", "fov_id"]).ngroups
    print(f"Loaded stats for {n_fovs_loaded} FOV×round combinations.")
    if "color" in stats_df.columns:
        colors = sorted(stats_df["color"].dropna().unique().astype(int))
        print(f"Colors found   : {colors}")

## 4 — Z-profiles

For each bit (round × color channel), plot median pixel intensity vs z position.  
Each line represents one FOV; lines are overlapping with α = 0.5.

In [ ]:
# Wavelength → plot color mapping (matches visualization.py)
_WL_COLOR = {
    405: "#9467bd",
    488: "#1f77b4",
    560: "#ff7f0e",
    650: "#2ca02c",
    750: "#d62728",
}
_DEFAULT_LINE_COLOR = "#7f7f7f"


def plot_z_profiles(stats_df, round_id, color_nm, bead_z=0.0, bead_color=488.0):
    """
    Z-profile plot for one bit (round_id × color_nm).
    One line per FOV; all lines share the same color, α = 0.5.
    """
    if "z" not in stats_df.columns or "color" not in stats_df.columns:
        print("No z/color information in stats (frame table not found).")
        return

    data = stats_df[
        (stats_df["round_id"] == round_id)
        & (stats_df["color"].round() == round(color_nm))
        & (stats_df["z"]    != bead_z)
    ].copy()

    if data.empty:
        print(f"No data for round {round_id}, color {color_nm:.0f} nm.")
        return

    line_color = _WL_COLOR.get(int(round(color_nm)), _DEFAULT_LINE_COLOR)

    fig, ax = plt.subplots(figsize=(7, 4))
    for _, fov_data in data.groupby("fov_id"):
        fov_data = fov_data.sort_values("z")
        ax.plot(fov_data["z"], fov_data["median"],
                color=line_color, alpha=0.5, linewidth=0.9)

    ax.set_xlabel("Z position (µm)")
    ax.set_ylabel("Median pixel intensity")
    ax.set_title(f"Round {round_id}  |  {color_nm:.0f} nm  —  Z-profile  "
                 f"({data['fov_id'].nunique()} FOVs)")
    fig.tight_layout()
    plt.show()


if stats_df.empty:
    print("No data to plot.")
elif "color" not in stats_df.columns:
    print("Column 'color' missing — frame table was not found for this experiment.")
else:
    rounds = sorted(stats_df["round_id"].unique())
    if SHOW_ROUNDS is not None:
        rounds = [r for r in rounds if r in SHOW_ROUNDS]

    for rid in rounds:
        rdata   = stats_df[
            (stats_df["round_id"] == rid)
            & (stats_df["z"] != BEAD_Z)
        ]
        colors  = sorted(rdata["color"].dropna().unique())
        colors  = [c for c in colors if round(c) != round(BEAD_COLOR)]
        if SHOW_COLORS is not None:
            colors = [c for c in colors if int(round(c)) in SHOW_COLORS]

        for c in colors:
            plot_z_profiles(stats_df, rid, c, BEAD_Z, BEAD_COLOR)

## 5 — FOV intensity heatmaps

For each bit, the median intensity of each FOV (median across z positions) is placed on a 2-D grid
derived from the stage positions.  
Brighter = higher signal.

In [ ]:
def _positions_to_grid_indices(fov_ids, meta):
    """
    Convert stage (x, y) positions to integer (x_idx, y_idx) grid indices.

    Stage positions lie on a regular grid with step ~200 µm.  Rounding to
    the nearest integer µm then ranking the unique values gives robust
    integer indices even with floating-point imprecision.

    Returns
    -------
    dict : {fov_id: (x_idx, y_idx)}
    """
    xs = np.array([round(meta.fovs[f].position[0]) for f in fov_ids])
    ys = np.array([round(meta.fovs[f].position[1]) for f in fov_ids])

    unique_xs = np.sort(np.unique(xs))
    unique_ys = np.sort(np.unique(ys))
    x_rank    = {v: i for i, v in enumerate(unique_xs)}
    y_rank    = {v: i for i, v in enumerate(unique_ys)}

    return {f: (x_rank[xs[i]], y_rank[ys[i]]) for i, f in enumerate(fov_ids)}


def plot_fov_heatmap(stats_df, meta, round_id, color_nm, bead_z=0.0, bead_color=488.0):
    """
    Heatmap of per-FOV median intensity (median over z) for one bit.

    Stage positions are mapped to integer (x_idx, y_idx) grid indices.
    The heatmap rows correspond to y_idx (increasing downward, matching
    the stage coordinate convention).
    """
    if "z" not in stats_df.columns or "color" not in stats_df.columns:
        print("No z/color information in stats (frame table not found).")
        return

    data = stats_df[
        (stats_df["round_id"] == round_id)
        & (stats_df["color"].round() == round(color_nm))
        & (stats_df["z"]    != bead_z)
    ]

    if data.empty:
        print(f"No data for round {round_id}, color {color_nm:.0f} nm.")
        return

    # Per-FOV median across z positions
    fov_medians = data.groupby("fov_id")["median"].median()
    fov_ids     = sorted(fov_medians.index.tolist())

    grid = _positions_to_grid_indices(fov_ids, meta)
    n_x  = max(xi for xi, _  in grid.values()) + 1
    n_y  = max(yi for _,  yi in grid.values()) + 1

    matrix = np.full((n_y, n_x), np.nan)
    for fov_id, intensity in fov_medians.items():
        xi, yi = grid[fov_id]
        matrix[yi, xi] = intensity

    fig, ax = plt.subplots(figsize=(max(5, n_x * 0.4 + 1.5),
                                    max(4, n_y * 0.4 + 1.5)))
    im = ax.imshow(matrix, cmap="viridis", origin="upper",
                   interpolation="nearest",
                   vmin=np.nanpercentile(matrix, 2),
                   vmax=np.nanpercentile(matrix, 98))
    cbar = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.04)
    cbar.set_label("Median pixel intensity")

    ax.set_title(f"Round {round_id}  |  {color_nm:.0f} nm  —  FOV intensity map  "
                 f"({len(fov_ids)} FOVs)")
    ax.set_xlabel("X grid index  (increasing stage X →)")
    ax.set_ylabel("Y grid index  (increasing stage Y ↓)")

    # Annotate each cell with its FOV id
    if n_x * n_y <= 200:   # only annotate if grid is not too large
        fov_at = {(xi, yi): fov_id for fov_id, (xi, yi) in grid.items()}
        for (xi, yi), fov_id in fov_at.items():
            intensity = matrix[yi, xi]
            txt_color = "white" if intensity < np.nanmedian(matrix) else "black"
            ax.text(xi, yi, str(fov_id), ha="center", va="center",
                    fontsize=6, color=txt_color)

    fig.tight_layout()
    plt.show()


if stats_df.empty:
    print("No data to plot.")
elif "color" not in stats_df.columns:
    print("Column 'color' missing — frame table was not found for this experiment.")
else:
    rounds = sorted(stats_df["round_id"].unique())
    if SHOW_ROUNDS is not None:
        rounds = [r for r in rounds if r in SHOW_ROUNDS]

    for rid in rounds:
        rdata  = stats_df[
            (stats_df["round_id"] == rid)
            & (stats_df["z"] != BEAD_Z)
        ]
        colors = sorted(rdata["color"].dropna().unique())
        colors = [c for c in colors if round(c) != round(BEAD_COLOR)]
        if SHOW_COLORS is not None:
            colors = [c for c in colors if int(round(c)) in SHOW_COLORS]

        for c in colors:
            plot_fov_heatmap(stats_df, meta, rid, c, BEAD_Z, BEAD_COLOR)